## 1. Load, Split Data

In [89]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset?select=spam.csv
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    'uciml/sms-spam-collection-dataset',
    'spam.csv',
    pandas_kwargs={"encoding": "latin-1"}
)

# Rename
df = df[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})

# Drop duplicates
df.drop_duplicates(subset='text')

# Check data
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [90]:
# Check spam / ham ratio
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [91]:
from sklearn.model_selection import train_test_split

# Assign feats, targets
X = df['text']
y = df['label']

# Split train, test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, stratify=y, random_state=42
)

# Check ratio after split
print(y_train.value_counts())
print(y_test.value_counts())

label
ham     3377
spam     523
Name: count, dtype: int64
label
ham     1448
spam     224
Name: count, dtype: int64


---

## 2. Encoding

In [92]:
from sklearn.preprocessing import LabelEncoder

# Transform spam, ham to 0, 1
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# See which index is ham
for i, name in enumerate(le.classes_):
    print(i, "=", name)

0 = ham
1 = spam


In [93]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier

# Init empty pipe
pipe = Pipeline([
    ('vec', None),
    ('model', DummyClassifier)
])

---

## 3. Vectorizers (Count, TF-IDF)

In [94]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Count vectorizer
count_params = {
    'vec': [CountVectorizer()],
    'vec__ngram_range': [(1, 1), (1, 2), (1, 3), (2, 3)],
    'vec__min_df': [0.001, 0.005, 0.01],
    'vec__max_df': [0.8, 0.9, 0.95, 1.0],
    'vec__stop_words': [None, 'english'],
    'vec__binary': [True, False],
    'vec__lowercase': [True],
    'vec__strip_accents': [None, 'unicode'],
    'vec__max_features': [None, 5000, 10000]
}

# TF-IDF vectorizer
tfidf_params = {
    **count_params,
    'vec': [TfidfVectorizer()],     # Overwrite count vectorizer
    'vec__sublinear_tf': [True, False],
    'vec__norm': [None, 'l2'],
    'vec__use_idf': [True, False],
    'vec__smooth_idf': [True, False]
}

---

## 4. Models

#### (LogisticRegression, LinearSVC, BernoulliNB, MultinomialNB, ComplementNB, RandomForest MLPClassifier)

In [95]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB, MultinomialNB, ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Logistic regression
logistic_params = {
    'model': [
        LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        )
    ],
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__solver': ['lbfgs', 'liblinear']
}

# Linear SVC
linear_svc_params = {
    'model': [
        LinearSVC(
            class_weight='balanced',
            max_iter=20000,
            random_state=42
        )
    ],
    'model__C': [0.01, 0.1, 1, 10]
}

# Bernoulli NB
bernoulli_params = {
    'model': [
        BernoulliNB()
    ],
    'model__alpha': [0.001, 0.01, 0.1, 1.0],
    'model__binarize': [0.0, 0.5],
    'model__fit_prior': [True, False]
}

# Multinomial NB
multinomial_params = {
    'model': [
        MultinomialNB()
    ],
    'model__alpha': [0.001, 0.01, 0.1, 1.0],
    'model__fit_prior': [True, False]
}

# Complement NB
complement_params = {
    'model': [
        ComplementNB()
    ],
    'model__alpha': [0.001, 0.01, 0.1, 1.0],
    'model__norm': [True, False]
}

# Random forest
rf_params = {
    'model': [
        RandomForestClassifier(
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
    ],
    'model__n_estimators': [100, 300],
    'model__max_depth': [None, 20, 50],
    'model__min_samples_leaf': [1, 2]
}

# MLP
mlp_params = {
    'model': [
        MLPClassifier(
            activation='relu',
            solver='adam',
            early_stopping=True,
            n_iter_no_change=10,
            random_state=42
        )
    ],
    'model__hidden_layer_sizes': [(20,), (20, 20), (20, 30, 20), (50, 50)],
    'model__learning_rate_init': [0.001, 0.01],
    'model__alpha': [0.0001, 0.001, 0.01, 0.1],
    'model__max_iter': [500, 1000]
}

---

## 5. Test Different Combinations

In [96]:
vec_model_params = []

# Generate all combinations
for vec_params in [count_params, tfidf_params]:
    for model_params in [
        logistic_params,
        linear_svc_params,
        bernoulli_params,
        multinomial_params,
        complement_params,
        rf_params,
        mlp_params
    ]:
        vec_model_params.append(vec_params | model_params)
vec_model_params[0:2]

[{'vec': [CountVectorizer()],
  'vec__ngram_range': [(1, 1), (1, 2), (1, 3), (2, 3)],
  'vec__min_df': [0.001, 0.005, 0.01],
  'vec__max_df': [0.8, 0.9, 0.95, 1.0],
  'vec__stop_words': [None, 'english'],
  'vec__binary': [True, False],
  'vec__lowercase': [True],
  'vec__strip_accents': [None, 'unicode'],
  'vec__max_features': [None, 5000, 10000],
  'model': [LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)],
  'model__C': [0.01, 0.1, 1, 10, 100],
  'model__solver': ['lbfgs', 'liblinear']},
 {'vec': [CountVectorizer()],
  'vec__ngram_range': [(1, 1), (1, 2), (1, 3), (2, 3)],
  'vec__min_df': [0.001, 0.005, 0.01],
  'vec__max_df': [0.8, 0.9, 0.95, 1.0],
  'vec__stop_words': [None, 'english'],
  'vec__binary': [True, False],
  'vec__lowercase': [True],
  'vec__strip_accents': [None, 'unicode'],
  'vec__max_features': [None, 5000, 10000],
  'model': [LinearSVC(class_weight='balanced', max_iter=20000, random_state=42)],
  'model__C': [0.01, 0.1, 1, 10]}]

In [97]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

best = None

# Init empty pipe
pipe = Pipeline([
    ('vec', None),
    ('model', DummyClassifier)
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Iterate combinations
for params in vec_model_params:
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params,
        cv=cv,
        scoring='f1',
        n_jobs=-1,
        n_iter=10,
        random_state=13,
        error_score='raise'
    )
    search.fit(X_train, y_train)

    # Keep track of best
    if best is None or search.best_score_ > best.best_score_:
        best = search

/Users/colinleong/Documents/Programming/spam-detection-api/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/colinleong/Documents/Programming/spam-detection-api/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


---

## 6. Results

In [98]:
print(f'Best CV F1: {best.best_score_:.4f}')
print(f'Best estimator:\n{best.best_estimator_}')
print(f'Best params:\n{best.best_params_}') 

Best CV F1: 0.9446
Best estimator:
Pipeline(steps=[('vec',
                 TfidfVectorizer(max_df=0.95, min_df=0.001, norm=None,
                                 smooth_idf=False, sublinear_tf=True)),
                ('model', BernoulliNB(fit_prior=False))])
Best params:
{'vec__use_idf': True, 'vec__sublinear_tf': True, 'vec__strip_accents': None, 'vec__stop_words': None, 'vec__smooth_idf': False, 'vec__norm': None, 'vec__ngram_range': (1, 1), 'vec__min_df': 0.001, 'vec__max_features': None, 'vec__max_df': 0.95, 'vec__lowercase': True, 'vec__binary': False, 'vec': TfidfVectorizer(), 'model__fit_prior': False, 'model__binarize': 0.0, 'model__alpha': 1.0, 'model': BernoulliNB()}


In [99]:
should_ham = search.predict(['Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'])
should_spam = search.predict(['SIX chances to win CASH! From 100 to 20,000 pounds txt> CSH11 and send to 87575. Cost 150p/day, 6days, 16+ TsandCs apply Reply HL 4 info'])

print(f'Should ham: {should_ham}')
print(f'Should spam: {should_spam}')

Should ham: [0]
Should spam: [1]
